In [32]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
import json
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from pathlib import Path
from tqdm import tqdm
from datasail.sail import datasail

from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

SEED = 42
np.random.seed(SEED)

In [33]:
class Config:
    TARGET_LIST = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    
    def get_canonical_smiles(self, smiles):
        for i in range(1, 10, 1):
            smiles = smiles.replace(f'[R{i}]', '[*]')
            
        smiles = smiles.replace('[R]', '[*]')
        smiles = smiles.replace('[R\']', '[*]')

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Invalid SMILES string: {smiles}")
        flag = Chem.SanitizeMol(mol, catchErrors=True)
        
        if flag != Chem.rdmolops.SanitizeFlags.SANITIZE_NONE:
            print(smiles)
            Chem.SanitizeMol(mol, sanitizeOps=Chem.rdmolops.SanitizeFlags.SANITIZE_ALL ^ flag)

        smiles = Chem.MolToSmiles(mol, canonical=True)
        
        return smiles
        
    def vectorize_smiles(self, smiles: str):
        smiles = self.get_canonical_smiles(smiles)
        mol = Chem.MolFromSmiles(smiles)
        useless_cols = [    
            'BCUT2D_MWHI',
            'BCUT2D_MWLOW',
            'BCUT2D_CHGHI',
            'BCUT2D_CHGLO',
            'BCUT2D_LOGPHI',
            'BCUT2D_LOGPLOW',
            'BCUT2D_MRHI',
            'BCUT2D_MRLOW',
            'NumRadicalElectrons',
            'SMR_VSA8',
            'SlogP_VSA9',
            'fr_barbitur',
            'fr_benzodiazepine',
            'fr_dihydropyridine',
            'fr_epoxide',
            'fr_isothiocyan',
            'fr_lactam',
            'fr_nitroso',
            'fr_prisulfonamd',
            'fr_thiocyan',
            'MaxEStateIndex',
            'HeavyAtomMolWt',
            'ExactMolWt',
            'NumValenceElectrons',
            'Chi0',
            'Chi0n',
            'Chi0v',
            'Chi1',
            'Chi1n',
            'Chi1v',
            'Chi2n',
            'Kappa1',
            'LabuteASA',
            'HeavyAtomCount',
            'MolMR',
            'Chi3n',
            'BertzCT',
            'Chi2v',
            'Chi4n',
            'HallKierAlpha',
            'Chi3v',
            'Chi4v',
            'MinAbsPartialCharge',
            'MaxPartialCharge',
            'MinPartialCharge',
            'MaxAbsPartialCharge',
            'FpDensityMorgan2',
            'FpDensityMorgan3',
            'Phi',
            'Kappa3',
            'fr_nitrile',
            'SlogP_VSA6',
            'NumAromaticCarbocycles',
            'NumAromaticRings',
            'fr_benzene',
            'VSA_EState6',
            'NOCount',
            'fr_C_O',
            'fr_C_O_noCOO',
            'NumHDonors',
            'fr_amide',
            'fr_Nhpyrrole',
            'fr_phenol',
            'fr_phenol_noOrthoHbond',
            'fr_COO2',
            'fr_halogen',
            'fr_diazo',
            'fr_nitro_arom',
            'fr_phos_ester'
        ]

        descriptors = {}
        # descriptors = {
        #     'MW': Descriptors.MolWt(mol),
        #     'HBA': Descriptors.NOCount(mol),
        #     'HBD': Descriptors.NHOHCount(mol),
        #     'LogP': Descriptors.MolLogP(mol),
        #     'TPSA': Descriptors.TPSA(mol),
        #     'EtherCount': Chem.Fragments.fr_ether(mol),
        #     'EsterCount': Chem.Fragments.fr_ester(mol),
        #     'AmideCount': Chem.Fragments.fr_amide(mol),
        #     'AromaticRingCount': Chem.rdMolDescriptors.CalcNumAromaticRings(mol),
        #     'BertzCT': Descriptors.BertzCT(mol),
        #     'BalabanJ': Descriptors.BalabanJ(mol)
        # }

        compute_desc = lambda mol: {nm: fn(mol) for nm, fn in Descriptors._descList if nm not in useless_cols}
        descriptors = compute_desc(mol)
        morgen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
        fp = morgen.GetFingerprint(mol)
        fp_feat = {idx: int(bit) for idx, bit in enumerate(fp)} 
        descriptors.update(fp_feat)
        return descriptors

config = Config()

In [34]:
path_to_data = 'data'
train_data = pd.read_csv(f'{path_to_data}/train.csv')
test_data = pd.read_csv(f'{path_to_data}/test.csv')

train_data = train_data.reset_index(drop=True)
d1 = pd.read_csv(f'{path_to_data}/dataset1.csv').reset_index(drop=True)
d3 = pd.read_csv(f'{path_to_data}/dataset3.csv').reset_index(drop=True)
d4 = pd.read_csv(f'{path_to_data}/dataset4.csv').reset_index(drop=True)


train_data = pd.concat([train_data, d1, d3, d4], axis=0).reset_index(drop=True)
train_data['id'] = train_data['id'].apply(lambda x: 0 if pd.isna(x) else x)
train_data['id'] = train_data['id'].astype(int)

data_dnst = pd.read_excel(f'{path_to_data}/data_dnst1.xlsx').iloc[:, [0,3]]
data_dnst = data_dnst.rename(columns={'density(g/cm3)': 'Density'}).reset_index(drop=True)
data_dnst['Density'] = pd.to_numeric(data_dnst['Density'], errors='coerce')
data_dnst = data_dnst.dropna(subset=['Density']).reset_index(drop=True)
data_dnst['Density'] = data_dnst['Density'].astype('float64')
data_dnst['Density'] -= 0.118

data_tg3 = pd.read_excel(f'{path_to_data}/data_tg3.xlsx')
data_tg3 = data_tg3.rename(columns={'Tg [K]' : 'Tg'}).reset_index(drop=True)
data_tg3['Tg'] = data_tg3['Tg'] - 273.15 

data_tg_ss = pd.read_csv(f'{path_to_data}/TgSS_enriched_cleaned.csv').iloc[:, [0,1]].reset_index(drop=True)

data_tc = pd.read_csv(f'{path_to_data}/Tc_SMILES.csv').iloc[:, [1,0]]
data_tc = data_tc.rename(columns={'TC_mean': 'Tc'}).reset_index(drop=True)

data_jcim = pd.read_csv(f'{path_to_data}/JCIM_sup_bigsmiles.csv').iloc[:, [1, -1]]
data_jcim = data_jcim.rename(columns={'Tg (C)': 'Tg'}).reset_index(drop=True)

train_data = pd.concat([train_data, data_dnst, data_tg3, data_tg_ss, data_tc, data_jcim], axis=0).reset_index(drop=True)
train_data['id'] = train_data['id'].apply(lambda x: 0 if pd.isna(x) else x)
train_data['id'] = train_data['id'].astype(int)

max_id = [train_data['id'].max()]
def increment_id(id):
    if id == 0:
        max_id[0] += 1
        return max_id[0]
    else:
        return id

train_data['id'] = train_data['id'].apply(increment_id)

In [35]:
errored = []

In [36]:
import re

def remove_radical_notation(smiles):
    smiles = re.sub(r"R[1-5']?", "*", smiles)

    return smiles

def clean_smiles(smiles):
    cleaned = remove_radical_notation(smiles)
    mol = Chem.MolFromSmiles(cleaned, sanitize=False)
    if mol is None:
        print(smiles)
        return None
    atoms_to_remove = []
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == '*' or atom.GetAtomicNum() == 0 or atom.GetIsotope() > 0 and atom.GetSymbol() == 'R':
            atoms_to_remove.append(atom.GetIdx())

    emol = Chem.EditableMol(mol)
    for idx in sorted(atoms_to_remove, reverse=True):
        emol.RemoveAtom(idx)
    new_mol = emol.GetMol()
    flag = Chem.SanitizeMol(mol, catchErrors=True)
        
    if flag != Chem.rdmolops.SanitizeFlags.SANITIZE_NONE:
        print(smiles)
        Chem.SanitizeMol(mol, sanitizeOps=Chem.rdmolops.SanitizeFlags.SANITIZE_ALL ^ flag)
        

    return Chem.MolToSmiles(new_mol)

In [37]:
train_data['corrected_smiles'] = train_data['SMILES'].apply(clean_smiles)

In [41]:
for target in config.TARGET_LIST:
    sub_df = train_data[train_data[target].notnull()].drop_duplicates(subset='SMILES', inplace=False, keep='first')
    
    e_splits, _, _ = datasail(
        techniques=["I1e"],
        splits=[8, 2],
        names=["train","val"],
        runs=5,
        threads=71,
        solver="SCIP",
        e_type="M",
        e_data=dict(sub_df[["id", "corrected_smiles"]].values.tolist())
    )
    with open(f'data/{target}_split.json', 'w') as f:
        json.dump(e_splits['I1e'], f)

100
                                     CVXPY                                     
                                     v1.5.3                                    
(CVXPY) Jul 27 03:20:17 PM: Your problem has 15752 variables, 7878 constraints, and 0 parameters.
(CVXPY) Jul 27 03:20:17 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 27 03:20:17 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 27 03:20:17 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 27 03:20:17 PM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Jul 27 03:20:17 PM: Compiling problem (target solver=S